# Node Cache（节点缓存）

> 适用版本：本项目锁定的 **LangGraph 1.1.2**。本笔记只使用该版本已经验证可用的 Graph API。

节点缓存根据节点的**实际输入**生成缓存键，保存节点成功执行产生的状态更新。以后同一节点再次收到相同缓存键时，LangGraph 可以跳过节点函数，直接重放缓存中的更新。它适合减少昂贵、确定性较强的计算或外部读取，例如 embedding、文档解析、稳定的模型调用结果或只读 API 查询。

节点缓存不是整张图的最终结果缓存：命中后，缓存节点的下游仍按正常图结构执行；只有也配置了缓存且命中的下游节点才会被跳过。

```mermaid
flowchart LR
    S([START]) --> N[准备节点输入]
    N --> K[key_func 生成缓存键]
    K --> C{缓存后端中存在且未过期?}
    C -->|命中| R[跳过节点函数并重放节点更新]
    C -->|未命中| E[执行节点函数]
    E -->|成功| W[按 TTL 写入缓存]
    E -->|抛出异常| X[异常处理或重试，不写缓存]
    W --> D[提交更新并执行下游]
    R --> D
    D --> F([END])
```

启用节点缓存需要两个配置同时存在：

1. 在目标节点上设置 `cache_policy=CachePolicy(...)`；
2. 编译图时提供缓存后端，例如 `compile(cache=InMemoryCache())`。

缺少其中任意一个，节点都不会真正复用缓存。

In [1]:
import time
from importlib.metadata import version
from typing import TypedDict

from langgraph.cache.memory import InMemoryCache
from langgraph.graph import END, START, StateGraph
from langgraph.types import CachePolicy, default_cache_key

print("LangGraph version:", version("langgraph"))


LangGraph version: 1.1.2


## 1. 基础示例：跨图调用复用节点结果

下面构建 `expensive_node → finalize` 两个节点：

- `expensive_node` 配置缓存，并用外部计数器模拟昂贵操作；
- `finalize` 不配置缓存，用来验证上游命中缓存后，下游仍然正常执行；
- `InMemoryCache` 对象在 `compile` 时传入，并由同一个已编译图的多次调用共享。

> 这里的全局计数器只用于教学观察，不是生产环境中的缓存命中统计方案。

In [2]:
class CacheState(TypedDict):
    x: int
    result: int
    final_result: str


expensive_call_count = 0
finalize_call_count = 0


def expensive_node(state: CacheState) -> dict[str, int]:
    """模拟一个值得缓存的确定性昂贵计算。"""

    global expensive_call_count
    expensive_call_count += 1
    print(
        f"expensive_node 真正执行：x={state['x']}，"
        f"累计调用 {expensive_call_count} 次"
    )
    return {"result": state["x"] * 2}


def finalize(state: CacheState) -> dict[str, str]:
    """未配置缓存的下游节点。"""

    global finalize_call_count
    finalize_call_count += 1
    print(f"finalize 真正执行：累计调用 {finalize_call_count} 次")
    return {"final_result": f"result={state['result']}"}


node_cache = InMemoryCache()

builder = StateGraph(state_schema=CacheState)
builder.add_node(
    "expensive_node",
    expensive_node,
    # ttl=None 表示不按时间过期；仍可能因后端清空或进程结束而消失。
    cache_policy=CachePolicy(ttl=None),
)
builder.add_node("finalize", finalize)
builder.add_edge(START, "expensive_node")
builder.add_edge("expensive_node", "finalize")
builder.add_edge("finalize", END)

cache_graph = builder.compile(cache=node_cache)


In [3]:
# 清理教学用缓存和计数器，让当前单元格可以安全地重复执行。
node_cache.clear()
expensive_call_count = 0
finalize_call_count = 0

# stream_mode="updates" 可以直接观察每个节点的更新和缓存命中元数据。
first_events = list(
    cache_graph.stream({"x": 5}, stream_mode="updates")
)
second_events = list(
    cache_graph.stream({"x": 5}, stream_mode="updates")
)
different_input_events = list(
    cache_graph.stream({"x": 6}, stream_mode="updates")
)

# clear() 不传 namespace 时清空该后端的全部缓存。
node_cache.clear()
after_clear_events = list(
    cache_graph.stream({"x": 5}, stream_mode="updates")
)

print("首次输入 x=5：", first_events)
print("再次输入 x=5：", second_events)
print("改为输入 x=6：", different_input_events)
print("清空后输入 x=5：", after_clear_events)
print(
    "真实调用次数：",
    {
        "expensive_node": expensive_call_count,
        "finalize": finalize_call_count,
    },
)

assert "__metadata__" not in first_events[0]
assert second_events[0]["__metadata__"] == {"cached": True}
assert "__metadata__" not in different_input_events[0]
assert "__metadata__" not in after_clear_events[0]
assert expensive_call_count == 3
assert finalize_call_count == 4


expensive_node 真正执行：x=5，累计调用 1 次
finalize 真正执行：累计调用 1 次
finalize 真正执行：累计调用 2 次
expensive_node 真正执行：x=6，累计调用 2 次
finalize 真正执行：累计调用 3 次
expensive_node 真正执行：x=5，累计调用 3 次
finalize 真正执行：累计调用 4 次
首次输入 x=5： [{'expensive_node': {'result': 10}}, {'finalize': {'final_result': 'result=10'}}]
再次输入 x=5： [{'expensive_node': {'result': 10}, '__metadata__': {'cached': True}}, {'finalize': {'final_result': 'result=10'}}]
改为输入 x=6： [{'expensive_node': {'result': 12}}, {'finalize': {'final_result': 'result=12'}}]
清空后输入 x=5： [{'expensive_node': {'result': 10}}, {'finalize': {'final_result': 'result=10'}}]
真实调用次数： {'expensive_node': 3, 'finalize': 4}


### 结果解读

- 第一次输入 `x=5` 时缓存未命中，两个节点都真正执行。
- 第二次输入 `x=5` 时，`expensive_node` 的事件带有 `{'__metadata__': {'cached': True}}`，函数没有再次运行；缓存中保存的 `{'result': 10}` 被当作本次节点更新重新提交。
- `finalize` 没有缓存策略，因此四次图调用中都执行。缓存命中不会直接跳到整张图的最终结果。
- 输入改成 `x=6` 后默认缓存键变化，所以重新计算；调用 `node_cache.clear()` 后，原输入也重新计算。
- `invoke()` 返回的最终 State 本身不附带通用的“缓存命中”字段；使用 `stream_mode="updates"` 可以在节点更新事件里看到命中元数据。

## 2. 只有 `CachePolicy` 还不够

`CachePolicy` 只描述“如何为这个节点生成键、缓存多久”，真正的读取和写入由图级缓存后端完成。如果 `compile()` 时没有传入 `cache`，LangGraph 1.1.2 不会报错，但也不会产生缓存命中。

In [4]:
missing_backend_calls = 0


def policy_only_node(state: CacheState) -> dict[str, int]:
    global missing_backend_calls
    missing_backend_calls += 1
    return {"result": state["x"] * 10}


policy_only_builder = StateGraph(state_schema=CacheState)
policy_only_builder.add_node(
    "policy_only_node",
    policy_only_node,
    cache_policy=CachePolicy(),
)
policy_only_builder.add_edge(START, "policy_only_node")
policy_only_builder.add_edge("policy_only_node", END)

# 故意不传 cache=...。
policy_only_graph = policy_only_builder.compile()
policy_only_graph.invoke({"x": 2})
policy_only_graph.invoke({"x": 2})

print("没有缓存后端时的真实调用次数：", missing_backend_calls)
assert missing_backend_calls == 2


没有缓存后端时的真实调用次数： 2


## 3. `CachePolicy` 参数与 TTL

`CachePolicy` 在 1.1.2 中只有两个参数：

| 参数 | 默认值 | 含义 |
| --- | --- | --- |
| `key_func` | `default_cache_key` | 接收节点输入并返回 `str` 或 `bytes`；默认将完整节点输入序列化后用于生成键 |
| `ttl` | `None` | 缓存存活秒数；`None` 表示不按时间自动过期 |

TTL 从成功结果写入缓存时开始计算。条目过期后，下一次相同输入会重新执行节点并覆盖缓存。下面使用 `ttl=1` 做实际验证。

In [5]:
ttl_call_count = 0
ttl_cache = InMemoryCache()


def expiring_node(state: CacheState) -> dict[str, int]:
    global ttl_call_count
    ttl_call_count += 1
    # 将真实调用序号写入结果，便于观察复用与重新计算。
    return {"result": ttl_call_count}


ttl_builder = StateGraph(state_schema=CacheState)
ttl_builder.add_node(
    "expiring_node",
    expiring_node,
    cache_policy=CachePolicy(ttl=1),
)
ttl_builder.add_edge(START, "expiring_node")
ttl_builder.add_edge("expiring_node", END)
ttl_graph = ttl_builder.compile(cache=ttl_cache)

first_result = ttl_graph.invoke({"x": 1})["result"]
cached_result = ttl_graph.invoke({"x": 1})["result"]
time.sleep(1.1)
expired_result = ttl_graph.invoke({"x": 1})["result"]

print(
    {
        "首次结果": first_result,
        "TTL 内结果": cached_result,
        "TTL 后结果": expired_result,
        "真实调用次数": ttl_call_count,
    }
)

assert (first_result, cached_result, expired_result) == (1, 1, 2)
assert ttl_call_count == 2


{'首次结果': 1, 'TTL 内结果': 1, 'TTL 后结果': 2, '真实调用次数': 2}


## 4. 自定义缓存键：只保留真正影响输出的输入

默认策略会根据节点收到的**完整输入**生成键。如果 State 中包含 `request_id`、追踪时间戳等与结果无关的字段，即使业务查询相同，也会形成不同键，降低命中率。

下面的节点输出只依赖规范化后的 `query`，因此自定义键会：

- 忽略与结果无关的 `request_id`；
- 对查询执行 `strip()` 和 `casefold()`；
- 加入 `normalize-query:v1` 版本前缀，便于算法、提示词或模型变化时主动失效旧缓存。

> 自定义键必须包含**所有会影响节点输出的依赖**。遗漏租户、模型、语言、权限或配置版本，可能返回过期结果，甚至把一个用户的结果错误地复用给另一个用户。

In [7]:
class QueryState(TypedDict):
    query: str
    request_id: str
    normalized: str


custom_key_call_count = 0
custom_key_cache = InMemoryCache()


def query_cache_key(state: QueryState) -> str:
    normalized_query = state["query"].strip().casefold()
    return f"normalize-query:v1:{normalized_query}"


def normalize_query(state: QueryState) -> dict[str, str]:
    global custom_key_call_count
    custom_key_call_count += 1
    return {"normalized": state["query"].strip().casefold()}


custom_key_builder = StateGraph(state_schema=QueryState)
custom_key_builder.add_node(
    "normalize_query",
    normalize_query,
    cache_policy=CachePolicy(key_func=query_cache_key, ttl=None),
)
custom_key_builder.add_edge(START, "normalize_query")
custom_key_builder.add_edge("normalize_query", END)
custom_key_graph = custom_key_builder.compile(cache=custom_key_cache)

input_a = {"query": " LangGraph " , "request_id": "req-001"}
input_b = {"query": "langgraph", "request_id": "req-002"}

# 默认键会看到 query 原始形式和 request_id，因此这两个输入不同。
assert default_cache_key(input_a) != default_cache_key(input_b)
# 自定义键只保留真正影响输出的规范化查询。
assert query_cache_key(input_a) == query_cache_key(input_b)

custom_first_events = list(
    custom_key_graph.stream(input_a, stream_mode="updates")
)
custom_second_events = list(
    custom_key_graph.stream(input_b, stream_mode="updates")
)

print("首次事件：", custom_first_events)
print("规范化后同键的第二次事件：", custom_second_events)
print("normalize_query 真实调用次数：", custom_key_call_count)

assert custom_key_call_count == 1
assert custom_second_events[0]["__metadata__"] == {"cached": True}
assert custom_second_events[0]["normalize_query"] == {
    "normalized": "langgraph"
}


首次事件： [{'normalize_query': {'normalized': 'langgraph'}}]
规范化后同键的第二次事件： [{'normalize_query': {'normalized': 'langgraph'}, '__metadata__': {'cached': True}}]
normalize_query 真实调用次数： 1


## 5. 运行时语义与容易忽略的边界

### 缓存的是什么

LangGraph 缓存的是节点成功执行产生的**写入/更新**，不是整张图的 State 快照。命中后，这些更新会像节点本次刚返回一样应用到 State，随后 reducer、边和下游节点继续正常工作。抛出异常的失败执行不会形成可复用的成功缓存；如果同时配置 `RetryPolicy`，节点先完成内部重试，最终成功的更新才可能写入缓存。

### 缓存键基于节点实际输入

对 `StateGraph` 节点而言，`key_func` 接收调度到该节点时的输入，可能包含上游已经写入的多个 State 字段，而不一定只包含最初传给 `graph.invoke()` 的字典。默认键不会自动理解节点到底读取了哪些字段。

节点的 `RunnableConfig`、runtime context 和外部数据库当前内容不会自动进入默认缓存键。如果节点输出依赖这些隐藏输入，应把必要的租户 ID、模型/提示词版本、权限范围或数据版本显式纳入节点输入和缓存键；做不到时不要缓存该节点。

### 缓存命中会跳过函数体

缓存节点函数不会在命中时执行，因此日志、计数、数据库写入、邮件发送等函数体副作用也不会发生。不要给“执行外部写操作”这种副作用节点配置结果缓存；节点缓存更适合纯计算或可接受短期陈旧的只读操作。

## 6. Cache、Checkpointer 与 Store 的区别

| 机制 | 核心目标 | 典型索引 | 主要行为 |
| --- | --- | --- | --- |
| Node Cache | 避免重复计算 | 节点身份 + 节点输入缓存键 | 跨图调用复用节点更新 |
| Checkpointer | 持久化执行状态 | `thread_id` + checkpoint | 在 superstep 边界保存 State，用于暂停、恢复、历史和容错 |
| Store | 保存业务长期记忆/数据 | namespace + key | 跨线程读取和写入应用数据 |

因此：

- 配置 checkpointer 不等于启用节点结果缓存；一次全新的图调用不会仅因输入相同就自动跳过昂贵节点。
- 配置 Node Cache 也不会产生可恢复的线程执行历史。
- `InMemoryCache` 与 `InMemorySaver` 是不同组件：前者缓存节点更新，后者保存 checkpoint。
- 同一个缓存后端可以跨多次调用复用结果；缓存键默认不包含 `thread_id`。多租户应用必须显式设计隔离字段。

## 7. 缓存失效与生命周期

常用失效手段：

1. **TTL**：适合允许短期陈旧的数据；到期后的下一次读取重新计算。
2. **版本化缓存键**：提示词、模型、算法或数据结构变化时，把 `v1` 改为 `v2`，自然避开旧条目。
3. **显式清空**：`InMemoryCache.clear()` 清空全部条目；生产后端可根据其 API 设计更精细的失效流程。
4. **更换缓存 namespace / 后端实例**：适合部署切换或大规模失效。

1.1.2 的缓存 namespace 会包含节点名称及函数的模块/限定名，但不会对函数实现源码做内容哈希。同名函数的实现、提示词或模型参数变化时，不应假设旧缓存会自动失效，因此版本化键非常重要。

本笔记使用的 `langgraph.cache.memory.InMemoryCache` 是当前进程内的教学/开发后端：进程结束后数据消失，也没有容量上限。它和 `langchain_core.caches.InMemoryCache`（主要用于 LangChain 模型缓存）不是同一个类。

## 8. 最佳实践与版本边界

### 最佳实践

1. **优先缓存确定性、昂贵、无副作用的节点**，例如纯计算、稳定文档解析和可容忍陈旧的只读查询。
2. **缓存键覆盖所有输出依赖**：业务输入、租户/用户、权限、语言、模型、提示词和数据版本缺一不可。
3. **不要把密钥或完整敏感内容直接拼进可观察的字符串键**；需要时使用稳定的非敏感标识或安全摘要，并保证后端隔离。
4. **为动态数据设置合理 TTL**，同时限制缓存容量并监控命中率、内存占用和陈旧结果。
5. **部署变更时主动失效**：使用版本前缀或清理流程，不依赖函数同名情况下的自动失效。
6. **缓存命中不是业务审计事件**：命中会跳过函数体；审计、计费、权限验证等不能只放在可能被缓存的节点内部。
7. **使用 `stream_mode="updates"` 或 tracing 验证命中**，不要仅根据响应变快推断缓存已生效。

### 1.1.2 与最新文档的差异

本项目的 1.1.2 已支持 `add_node(..., cache_policy=CachePolicy(...))` 和 `compile(cache=...)`，也是本笔记全部示例的基础。较新的官方文档还介绍 `StateGraph.set_node_defaults(cache_policy=...)`；该方法需要 LangGraph 1.2，当前环境没有，因此 1.1.2 中应逐个节点声明缓存策略。

### 参考资料

- [LangGraph 官方 Graph API：Node caching](https://docs.langchain.com/oss/python/langgraph/graph-api#node-caching)
- [LangGraph 1.1.2 `CachePolicy` 源码](https://github.com/langchain-ai/langgraph/blob/1.1.2/libs/langgraph/langgraph/types.py)
- [LangGraph 1.1.2 `InMemoryCache` 源码](https://github.com/langchain-ai/langgraph/blob/1.1.2/libs/langgraph/langgraph/cache/memory/__init__.py)
- [LangGraph 1.1.2 缓存键与任务调度源码](https://github.com/langchain-ai/langgraph/blob/1.1.2/libs/langgraph/langgraph/pregel/_algo.py)
